# 171 — Weight Correlation Analysis

Analyze correlations in model weights: cross-layer similarity,
norm patterns, head-to-head similarity, QK/OV balance,
and deviation from initialization.

## Why This Matters

Weight correlation examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_correlation_analysis import (
    cross_layer_weight_correlation,
    weight_norm_pattern,
    head_weight_similarity,
    qk_ov_weight_balance,
    weight_initialization_deviation,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

In [ ]:
for mt in ['W_Q', 'W_K', 'W_V', 'W_O']:
    result = cross_layer_weight_correlation(model, mt)
    print(f"{mt}: mean_corr={result['mean_correlation']:.4f}")

In [ ]:
result = weight_norm_pattern(model)
print(f"Norm trend: {result['norm_trend']:+.4f}\n")
for p in result['per_layer']:
    norms_str = '  '.join(f"{k}={v:.3f}" for k, v in p['norms'].items())
    print(f"Layer {p['layer']}: total={p['total_norm']:.3f}  {norms_str}")

In [ ]:
for l in range(4):
    result = head_weight_similarity(model, layer=l)
    print(f"\nLayer {l}:")
    for p in result['per_pair']:
        print(f"  H{p['head_i']}<->H{p['head_j']}: Q={p['q_similarity']:.3f}  "
              f"K={p['k_similarity']:.3f}  V={p['v_similarity']:.3f}  mean={p['mean_similarity']:.3f}")

In [ ]:
result = qk_ov_weight_balance(model)
for l in result['per_layer']:
    for h in l['per_head']:
        bar_qk = '#' * int(h['qk_fraction'] * 20)
        bar_ov = '#' * int((1-h['qk_fraction']) * 20)
        print(f"L{l['layer']}H{h['head']}: QK={h['qk_norm']:.3f}[{bar_qk}]  "
              f"OV={h['ov_norm']:.3f}[{bar_ov}]")

In [ ]:
result = weight_initialization_deviation(model)
for p in result['per_layer']:
    print(f"Layer {p['layer']}:")
    for name, dev in p['deviations'].items():
        print(f"  {name}: mean={dev['mean']:+.4f}  std={dev['std']:.4f}  kurtosis={dev['kurtosis']:+.2f}")